# Task 7: Machine Translation - Finetuned Model (Practical Pipeline)

## Objective
Finetune a pretrained model for production-quality EN↔NL round-translation.

## Why This Approach?
- The from-scratch model (main notebook) demonstrates understanding.
- This finetuned model provides practical translation quality for the pipeline.
- Combines educational learning with real-world application.

---

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import (
    MarianMTModel, 
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch
from nltk.translate.bleu_score import corpus_bleu
import random

c:\Users\Erik\anaconda3\envs\year2a\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

## Step 1: Load pretrained models

In [3]:
# Load pretrained models
print('Loading pretrained models...')

# EN -> NL model
model_en_nl_name = "Helsinki-NLP/opus-mt-en-nl"
tokenizer_en_nl = MarianTokenizer.from_pretrained(model_en_nl_name)
model_en_nl = MarianMTModel.from_pretrained(model_en_nl_name)

# NL -> EN model
model_nl_en_name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer_nl_en = MarianTokenizer.from_pretrained(model_nl_en_name)
model_nl_en = MarianMTModel.from_pretrained(model_nl_en_name)

print('Models loaded')

Loading pretrained models...


c:\Users\Erik\anaconda3\envs\year2a\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Models loaded


c:\Users\Erik\anaconda3\envs\year2a\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Erik\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-nl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Erik\anaconda3\envs\year2a\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `h

## Step 2: Zero-shot test

In [8]:
def translate_marian(text, model, tokenizer, max_length=128):
    """ Translate using MarianMT model. """
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=max_length)
    translated = model.generate(**inputs, max_length=max_length, num_beams=5)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)
    return result

In [9]:
test_sentences = [
    'Hello, how are you?',
    'I love cooking.',
    'The weather is nice today.'
]

for sent in test_sentences:
    nl_translation = translate_marian(sent, model_en_nl, tokenizer_en_nl)
    en_translation = translate_marian(nl_translation, model_nl_en, tokenizer_nl_en)

    print(f'Original: {sent}')
    print(f'NL: {nl_translation}')
    print(f'Back EN: {en_translation}')
    print()

Original: Hello, how are you?
NL: Hallo, hoe gaat het?
Back EN: Hello, how are you?

Original: I love cooking.
NL: Ik hou van koken.
Back EN: I like to cook.

Original: The weather is nice today.
NL: Het weer is mooi vandaag.
Back EN: The weather's nice today.



## Step 3: Load Training Data


In [10]:
# Load the OpenSubtitles data
with open('../Data/Translation/OpenSubtitles.en-nl.en', 'r', encoding='utf-8') as f:
    en_sentences = [line.strip() for line in f.readlines()]

with open('../Data/Translation/OpenSubtitles.en-nl.nl', 'r', encoding='utf-8') as f:
    nl_sentences = [line.strip() for line in f.readlines()]

In [11]:
# Sample to manageable size
SAMPLE_SIZE = 50000 

# Sample randomly to get diverse data
indices = random.sample(range(len(en_sentences)), SAMPLE_SIZE)
en_sentences = [en_sentences[i] for i in sorted(indices)]
nl_sentences = [nl_sentences[i] for i in sorted(indices)]

print(f'Sampled {len(en_sentences)} sentence pairs')

Sampled 50000 sentence pairs


## Step 4: Prepare Dataset for Fine-Tuning

In [12]:
from sklearn.model_selection import train_test_split

# Split data
train_en, val_en, train_nl, val_nl = train_test_split(
    en_sentences, nl_sentences, test_size=0.1, random_state=42
)

# Create datasets for EN→NL
train_data_en_nl = Dataset.from_dict({
    'en': train_en,
    'nl': train_nl
})

val_data_en_nl = Dataset.from_dict({
    'en': val_en,
    'nl': val_nl
})

# Create datasets for NL→EN
train_data_nl_en = Dataset.from_dict({
    'nl': train_nl,
    'en': train_en
})

val_data_nl_en = Dataset.from_dict({
    'nl': val_nl,
    'en': val_en
})

print(f'Training samples: {len(train_en)}')
print(f'Validation samples: {len(val_en)}')

Training samples: 45000
Validation samples: 5000


## Step 5: Tokenization

In [13]:
def preprocess_function_en_nl(sentences):
    """ Tokenize for EN->NL model """
    inputs = tokenizer_en_nl(sentences['en'], truncation=True, max_length=128)
    targets = tokenizer_en_nl(sentences['nl'], truncation=True, max_length=128)
    inputs['labels'] = targets['input_ids']
    return inputs

def preprocess_function_nl_en(examples):
    """ Tokenize for NL->EN model """
    inputs = tokenizer_nl_en(examples['nl'], truncation=True, max_length=128)
    targets = tokenizer_nl_en(examples['en'], truncation=True, max_length=128)
    inputs['labels'] = targets['input_ids']
    return inputs

In [14]:
# Tokenize datasets
train_dataset_en_nl = train_data_en_nl.map(preprocess_function_en_nl, batched=True)
val_dataset_en_nl = val_data_en_nl.map(preprocess_function_en_nl, batched=True)

train_dataset_nl_en = train_data_nl_en.map(preprocess_function_nl_en, batched=True)
val_dataset_nl_en = val_data_nl_en.map(preprocess_function_nl_en, batched=True)

print('Datasets tokenized')

Map: 100%|██████████| 5000/5000 [00:00<00:00, 7176.41 examples/s]

Datasets tokenized


## Step 6: Fine-Tuning EN->NL model

In [ ]:
# Training arguments
training_args_en_nl = Seq2SeqTrainingArguments(
    output_dir='./machine_translation/results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False,
    logging_dir='./machine_translation/logs',
    logging_steps=100,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Data collator
data_collator_en_nl = DataCollatorForSeq2Seq(tokenizer_en_nl, model=model_en_nl)

# Trainer
trainer_en_nl = Seq2SeqTrainer(
    model=model_en_nl,
    args=training_args_en_nl,
    train_dataset=train_dataset_en_nl,
    eval_dataset=val_dataset_en_nl,
    data_collator=data_collator_en_nl,
    tokenizer=tokenizer_en_nl,
)

C:\Users\Erik\AppData\Local\Temp\ipykernel_15988\1844505928.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_en_nl = Seq2SeqTrainer(


In [ ]:
print('Starting training...')
trainer_en_nl.train()

# Save finetuned model
model_en_nl.save_pretrained('./machine_translation/finetuned_en_nl')
tokenizer_en_nl.save_pretrained('./machine_translation/finetuned_en_nl')
print('EN->NL model finetuned and saved')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Step 7: Fine-Tuning NL->EN Model

In [ ]:
# Training arguments
training_args_nl_en = Seq2SeqTrainingArguments(
    output_dir='./machine_translation/results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False,
    logging_dir='./machine_translation/logs',
    logging_steps=100,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Data collator
data_collator_nl_en = DataCollatorForSeq2Seq(tokenizer_nl_en, model=model_nl_en)

# Trainer
trainer_nl_en = Seq2SeqTrainer(
    model=model_nl_en,
    args=training_args_nl_en,
    train_dataset=train_dataset_nl_en,
    eval_dataset=val_dataset_nl_en,
    data_collator=data_collator_nl_en,
    tokenizer=tokenizer_nl_en,
)

In [ ]:
print('Starting training...')
trainer_nl_en.train()

# Save finetuned model
model_nl_en.save_pretrained('./machine_translation/finetuned_nl_en')
tokenizer_nl_en.save_pretrained('./machine_translation/finetuned_nl_en')
print('NL->EN model finetuned and saved')

## Step 8: Evaluation

In [ ]:
def evaluate_marian_bleu(en_texts, nl_texts, direction='en_to_nl', n_samples=500):
    """ Evaluate BLEU for MarianMT models """
    references = []
    hypotheses = []
    
    samples = min(n_samples, len(en_texts))
    
    if direction == 'en_to_nl':
        model, tokenizer = model_en_nl, tokenizer_en_nl
        sources = en_texts[:samples]
        targets = nl_texts[:samples]
    else:
        model, tokenizer = model_nl_en, tokenizer_nl_en
        sources = nl_texts[:samples]
        targets = en_texts[:samples]
    
    print(f'Evaluating {samples} samples for {direction}...')
    
    for i in range(samples):
        ref_tokens = targets[i].strip().split()
        translation = translate_marian(sources[i], model, tokenizer)
        hyp_tokens = translation.strip().split()
        
        references.append([ref_tokens])
        hypotheses.append(hyp_tokens)
        
        if (i + 1) % 100 == 0:
            print(f'  Progress: {i+1}/{samples}')
    
    bleu = corpus_bleu(references, hypotheses)
    return bleu * 100

In [ ]:
# Evaluate finetuned models
bleu_en_nl = evaluate_marian_bleu(val_en, val_nl, direction='en_to_nl')
bleu_nl_en = evaluate_marian_bleu(val_en, val_nl, direction='nl_to_en')

print(f'BLEU EN->NL: {bleu_en_nl:.2f}')
print(f'BLEU NL->EN: {bleu_nl_en:.2f}')
print(f'Average: {(bleu_en_nl + bleu_nl_en) / 2:.2f}')